# Lab 1 — Tokenization Explorer

## Week 1: Foundations of LLM Inference

This notebook helps you investigate how raw text becomes tokens, token IDs, and model-ready numerical input.

The main lesson:

> Tokens are the operational currency of LLM inference systems.

They affect:
- latency
- memory usage
- context window pressure
- API cost
- throughput
- batching
- KV cache growth


## 1. Install Dependencies

Run this cell if you are using Google Colab or a fresh environment.

If you already installed dependencies locally, you can skip it.


In [ ]:
# Uncomment if needed
# !pip install transformers torch accelerate tabulate pandas matplotlib

## 2. Load the Tokenizer

We use a small Qwen tokenizer because it aligns with the larger Qwen-family capstone later in the course.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loaded tokenizer: {MODEL_NAME}")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")

## 3. Basic Tokenization

A language model does not receive raw text directly.

It receives token IDs.

This cell shows:
- raw text
- token pieces
- token IDs
- token count


In [ ]:
texts = [
    "Hello world",
    "The capital of Ghana is Accra.",
    "Inference engineering is about systems, memory, latency, and cost.",
    "microarchitectural optimization",
    "Akwaaba! How are you doing today?",
]

for text in texts:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)

    print("=" * 80)
    print("TEXT:")
    print(text)

    print("\nTOKENS:")
    print(tokens)

    print("\nTOKEN IDS:")
    print(token_ids)

    print("\nTOKEN COUNT:")
    print(len(token_ids))

## 4. Words vs Tokens

A common beginner mistake is assuming:

```text
1 word = 1 token
```

That is false.

Let’s compare word count and token count.


In [ ]:
import pandas as pd

rows = []

for text in texts:
    word_count = len(text.split())
    token_count = len(tokenizer.encode(text))
    ratio = token_count / word_count if word_count else 0

    rows.append({
        "text": text,
        "word_count": word_count,
        "token_count": token_count,
        "tokens_per_word": round(ratio, 2),
    })

df = pd.DataFrame(rows)
df

## 5. Subword Splitting

Technical words, rare words, names, code, emojis, and multilingual text may tokenize inefficiently.

This matters because more tokens mean more inference cost.


In [ ]:
test_cases = [
    "microarchitectural",
    "autoregressive",
    "internationalization",
    "quantization",
    "Shadrack Darku",
    "Kwame Nkrumah Circle",
    "def hello_world(): print('hello')",
    "😂😂😂😂😂",
    "Akwaaba, yɛfrɛ me Shadrack.",
]

rows = []

for text in test_cases:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)
    word_count = len(text.split())
    token_count = len(token_ids)

    rows.append({
        "text": text,
        "word_count": word_count,
        "token_count": token_count,
        "tokens_per_word": round(token_count / word_count, 2) if word_count else token_count,
        "tokens": tokens,
    })

pd.DataFrame(rows)

## 6. Prompt Style Comparison

Two prompts may ask for a similar thing but create very different token loads.

This affects:
- prefill latency
- context window usage
- KV cache allocation
- serving cost


In [ ]:
prompt_a = "Explain inference."

prompt_b = (
    "Please provide a detailed technical explanation of transformer inference, "
    "including tokenization, prefill, decode, KV cache, latency, throughput, "
    "and production serving tradeoffs."
)

for label, prompt in [("Prompt A", prompt_a), ("Prompt B", prompt_b)]:
    tokens = tokenizer.tokenize(prompt)
    token_ids = tokenizer.encode(prompt)

    print("=" * 80)
    print(label)
    print(prompt)
    print("\nToken count:", len(token_ids))
    print("Tokens:", tokens)

## 7. Simple Token Cost Estimator

This is not exact pricing. It is a teaching tool.

The goal is to understand that token count scales operational cost.


In [ ]:
def estimate_token_usage_cost(input_tokens, output_tokens, input_price_per_million=0.15, output_price_per_million=0.60):
    input_cost = (input_tokens / 1_000_000) * input_price_per_million
    output_cost = (output_tokens / 1_000_000) * output_price_per_million
    return input_cost + output_cost

example_prompts = [
    "Explain inference.",
    prompt_b,
    " ".join(["This is a long synthetic prompt used to simulate context pressure."] * 100),
]

rows = []

for prompt in example_prompts:
    input_tokens = len(tokenizer.encode(prompt))
    assumed_output_tokens = 300
    estimated_cost = estimate_token_usage_cost(input_tokens, assumed_output_tokens)

    rows.append({
        "prompt_preview": prompt[:60] + "...",
        "input_tokens": input_tokens,
        "assumed_output_tokens": assumed_output_tokens,
        "estimated_cost_usd": round(estimated_cost, 6),
    })

pd.DataFrame(rows)

## 8. Operational Interpretation

A strong inference engineer does not stop at:

> The tokenizer produced tokens.

They ask:

- What does this imply for latency?
- What does this imply for memory?
- What does this imply for cost?
- What does this imply for context window pressure?


In [ ]:
def operational_interpretation(token_count):
    if token_count > 500:
        return "High token load: likely significant prefill cost, context pressure, and KV cache growth."
    elif token_count > 100:
        return "Moderate token load: noticeable impact on prefill latency and memory usage."
    elif token_count > 20:
        return "Small-to-moderate token load: manageable but still relevant at scale."
    else:
        return "Token-efficient prompt: low relative prefill overhead."

for prompt in example_prompts:
    token_count = len(tokenizer.encode(prompt))
    print("=" * 80)
    print("Prompt preview:", prompt[:100])
    print("Token count:", token_count)
    print("Interpretation:", operational_interpretation(token_count))

## 9. Student Exercise

Add your own examples below.

Try:
- Ghanaian names
- Twi or mixed-language text
- code snippets
- emojis
- legal/business text
- short vs long prompts

Then explain which inputs are least token-efficient.


In [ ]:
my_examples = [
    # Add your examples here
    "Replace this with your own example.",
]

for text in my_examples:
    tokens = tokenizer.tokenize(text)
    token_ids = tokenizer.encode(text)

    print("=" * 80)
    print("TEXT:", text)
    print("TOKEN COUNT:", len(token_ids))
    print("TOKENS:", tokens)
    print("INTERPRETATION:", operational_interpretation(len(token_ids)))

## 10. Reflection Questions

Answer these in your lab report.

1. What surprised you most about the tokenizer output?
2. Which input was least token-efficient?
3. Why is token count more useful than word count for inference engineering?
4. How does token count affect TTFT?
5. How does token count affect context window pressure?
6. Why might multilingual systems have different serving costs?
7. What did this lab teach you about production LLM systems?


## Key Lesson

Tokens are not just preprocessing artifacts.

Tokens are the economic, computational, and memory unit of LLM inference systems.
